# КЭП · «Космос как инфраструктура» — стартовый framework

**Это не эталонное решение.** Notebook реализует только канонический пересчет и проверки. Метод выбора и управленческую логику создает команда.

Для запуска в Google Colab достаточно выбрать **Среда выполнения → Выполнить все**. Первая кодовая ячейка сама подключит публичный репозиторий кейса, если файлы не найдены локально.

In [ ]:
# 0. Среда: one-click Colab или локальный clone репозитория.
from pathlib import Path
import sys, json, subprocess, shutil, pandas as pd, numpy as np

REPO_URL = "https://github.com/SpaceEconomyPolicy/test.git"
CLONE_DIR = Path('/content/kep_case')
REQUIRED = [
    Path('case_core.py'),
    Path('data')/'lots.csv',
    Path('data')/'access_modes.csv',
    Path('config')/'case_config.json',
]

def is_case_root(path):
    path = Path(path)
    return all((path / rel).exists() for rel in REQUIRED)

def find_local_root():
    candidates = [Path.cwd(), Path.cwd().parent, CLONE_DIR]
    for candidate in candidates:
        if is_case_root(candidate):
            return candidate.resolve()
    return None

ROOT = find_local_root()
if ROOT is None:
    if CLONE_DIR.exists():
        shutil.rmtree(CLONE_DIR)
    print('Файлы кейса не найдены локально — клонирую:', REPO_URL)
    subprocess.run(['git','clone','--depth','1',REPO_URL,str(CLONE_DIR)], check=True)
    ROOT = CLONE_DIR.resolve()
    missing = [str(rel) for rel in REQUIRED if not (ROOT / rel).exists()]
    if missing:
        raise FileNotFoundError(
            'Репозиторий загружен, но не хватает обязательных файлов: ' + ', '.join(missing)
        )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from case_core import load_case, evaluate_portfolio, check_constraints
lots, modes, cfg = load_case(ROOT)
print('Case version:', cfg['case_version'], '| root:', ROOT, '| lots:', len(lots), '| modes:', len(modes))
display(lots[['lot_id','territorial_archetype','service','capability_groups','c0_mrub','opex_mrub_per_year','vpub_mrub_per_year','t_rep']])


## 1. Карточка решения команды
Заполните формы. В Colab поля `#@param` отображаются как UI-контролы.

In [ ]:
team_name = "" #@param {type:"string"}
decision_method = "Weighted MCDA" #@param ["Weighted MCDA","Pareto/frontier","Rule-based/manual","Optimization","Other"]
strategy_thesis = "" #@param {type:"string"}
print(team_name or 'Команда не указана', '|', decision_method)

## 2. Опциональный собственный режим доступа
Авторский режим — не способ подобрать коэффициенты под желаемый ответ. Если используете `D`, задайте все коэффициенты, обоснуйте причинную связь и проведите чувствительность. Если `D` считается Public Core, базовый общественно значимый слой должен оставаться бесплатным/недискриминационным.

In [ ]:
enable_custom_mode = False #@param {type:"boolean"}
custom_mode_name = "Team-defined" #@param {type:"string"}
k_c0_D = 1.00 #@param {type:"number"}
k_opex_D = 1.00 #@param {type:"number"}
k_vpub_D = 0.80 #@param {type:"number"}
k_anchor_D = 0.80 #@param {type:"number"}
k_commercial_D = 0.80 #@param {type:"number"}
custom_public_core = False #@param {type:"boolean"}
custom_mode_rationale = "" #@param {type:"string"}

modes_work = modes.copy()
if enable_custom_mode:
    row={'mode_id':'D','name_ru':custom_mode_name,'k_c0':k_c0_D,'k_opex':k_opex_D,'k_vpub':k_vpub_D,'k_anchor':k_anchor_D,'k_commercial':k_commercial_D,'public_core':custom_public_core,'description':custom_mode_rationale}
    modes_work=pd.concat([modes_work,pd.DataFrame([row])],ignore_index=True)
    print('Добавлен режим D. Зафиксируйте rationale и sensitivity в материалах команды.')
else:
    print('Используются только канонические A/B/C.')

## 3. Выбор 4 лотов и режимов
Оставьте пустое значение до того, как определитесь. Один лот нельзя выбирать дважды.

In [ ]:
lot_1 = "" #@param ["","FIRE","FLOOD","AGRI","INFRA","ARCTIC","TRANS","ENV","SSA"]
mode_1 = "A" #@param ["A","B","C","D"]
lot_2 = "" #@param ["","FIRE","FLOOD","AGRI","INFRA","ARCTIC","TRANS","ENV","SSA"]
mode_2 = "A" #@param ["A","B","C","D"]
lot_3 = "" #@param ["","FIRE","FLOOD","AGRI","INFRA","ARCTIC","TRANS","ENV","SSA"]
mode_3 = "A" #@param ["A","B","C","D"]
lot_4 = "" #@param ["","FIRE","FLOOD","AGRI","INFRA","ARCTIC","TRANS","ENV","SSA"]
mode_4 = "A" #@param ["A","B","C","D"]
selection=[(l,m) for l,m in [(lot_1,mode_1),(lot_2,mode_2),(lot_3,mode_3),(lot_4,mode_4)] if l]
print('Selection:', selection)
if len({l for l,_ in selection}) != len(selection): print('⚠ Один лот выбран более одного раза.')

## 4. Канонический расчет BASE/STRESS

In [ ]:
if len(selection)==4 and len({x[0] for x in selection})==4:
    detail, metrics = evaluate_portfolio(selection, lots, modes_work, cfg)
    display(detail[['lot_id','mode_id','c0_mrub','opex_mrub_per_year','vpub_mrub_per_year','cash_mrub_per_year','t_rep']].round(3))
    display(pd.DataFrame([metrics]).round(3))
    print('BASE')
    display(check_constraints(metrics,cfg,'BASE'))
    print('STRESS')
    display(check_constraints(metrics,cfg,'STRESS'))
else:
    print('Выберите 4 уникальных лота. Расчет будет выполнен после заполнения формы.')

## 5. Собственная decision model (опционально)
Ниже — **стартовая форма**, а не обязательные веса. Если используете MCDA, объясните нормализацию, веса и чувствительность. Если используете Парето/правила/оптимизацию — адаптируйте блок.

In [ ]:
w_vpub = 0.30 #@param {type:"number"}
w_capex = 0.15 #@param {type:"number"}
w_opex = 0.10 #@param {type:"number"}
w_kcash = 0.15 #@param {type:"number"}
w_trep = 0.10 #@param {type:"number"}
w_readiness = 0.07 #@param {type:"number"}
w_resilience = 0.08 #@param {type:"number"}
w_scale = 0.05 #@param {type:"number"}
weights={'vpub':w_vpub,'capex':w_capex,'opex':w_opex,'kcash':w_kcash,'t_rep':w_trep,'readiness':w_readiness,'resilience':w_resilience,'scale':w_scale}
print('Сумма весов:', round(sum(weights.values()),6))
if decision_method=='Weighted MCDA' and abs(sum(weights.values())-1)>1e-9:
    print('⚠ Для weighted MCDA нормализуйте веса или обоснуйте другую схему.')

## 6. Sensitivity hook
Минимум проверьте факторы, которые реально могут изменить ваш управленческий выбор. Пример ниже создает ±20% диапазон для двух выбранных весов, но **не выбирает портфель автоматически**.

In [ ]:
sensitivity_weight_1 = "vpub" #@param ["vpub","capex","opex","kcash","t_rep","readiness","resilience","scale"]
sensitivity_weight_2 = "kcash" #@param ["vpub","capex","opex","kcash","t_rep","readiness","resilience","scale"]
for key in [sensitivity_weight_1,sensitivity_weight_2]:
    base=weights[key]
    print(key, 'range:', round(base*0.8,4), '→', round(base*1.2,4))
print('Добавьте функцию team_score(...) или Парето-анализ в соответствии с вашим методом.')

## 7. Управленческие поля
Расчет сам по себе не отвечает на вопрос кейса.

In [ ]:
payer_opex = "" #@param {type:"string"}
operator_model = "" #@param {type:"string"}
supplier_switch_rule = "" #@param {type:"string"}
replicable_core = "" #@param {type:"string"}
local_adaptation = "" #@param {type:"string"}
stress_decision = "" #@param {type:"string"}
print('Заполнено управленческих полей:', sum(bool(x.strip()) for x in [payer_opex,operator_model,supplier_switch_rule,replicable_core,local_adaptation,stress_decision]), '/ 6')

## 8. Экспорт результатов
После полного заполнения сохраните воспроизводимые цифры рядом с запиской.

In [ ]:
from pathlib import Path
out=Path('results'); out.mkdir(exist_ok=True)
if len(selection)==4 and len({x[0] for x in selection})==4:
    detail, metrics = evaluate_portfolio(selection, lots, modes_work, cfg)
    detail.to_csv(out/'portfolio_detail.csv',index=False)
    with open(out/'portfolio_metrics.json','w',encoding='utf-8') as f: json.dump(metrics,f,ensure_ascii=False,indent=2)
    summary={'team':team_name,'decision_method':decision_method,'strategy_thesis':strategy_thesis,'selection':selection,'weights':weights,'management':{'payer_opex':payer_opex,'operator_model':operator_model,'supplier_switch_rule':supplier_switch_rule,'replicable_core':replicable_core,'local_adaptation':local_adaptation,'stress_decision':stress_decision}}
    with open(out/'team_decision_config.json','w',encoding='utf-8') as f: json.dump(summary,f,ensure_ascii=False,indent=2)
    print('Сохранено в', out.resolve())
else:
    print('Экспорт появится после выбора 4 уникальных лотов.')